# Prompts for Specific Tasks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/specific-task-prompts.ipynb)

## Overview

This tutorial explores the creation and use of prompts for specific tasks in natural language processing. We'll focus on four key areas: text summarization, question-answering, code generation, and creative writing. Using the **Qwen3-8B** open-source model running locally via HuggingFace Transformers on Google Colab, we'll demonstrate how to craft effective prompts for each of these tasks.

## Motivation

As language models become more advanced, the ability to design task-specific prompts becomes increasingly valuable. Well-crafted prompts can significantly enhance the performance of AI models across various applications, from summarizing long documents to generating code and fostering creativity in writing. This tutorial aims to provide practical insights into prompt engineering for these diverse tasks.

## Key Components

1. Text Summarization Prompts: Techniques for condensing long texts while retaining key information.
2. Question-Answering Prompts: Strategies for extracting specific information from given contexts.
3. Code Generation Prompts: Methods for guiding AI models to produce accurate and functional code.
4. Creative Writing Prompts: Approaches to stimulating imaginative and engaging written content.

## Method Details

This tutorial uses the **Qwen3-8B** model loaded in 4-bit quantization via HuggingFace Transformers. For each task type, we'll follow these steps:

1. Design a prompt using the chat-message format (system + user messages).
2. Implement the prompt using Python f-strings and a reusable `generate()` helper.
3. Tune **generation parameters** (temperature, max tokens) to match the task.
4. Execute the prompt with sample inputs and analyze the output.

We'll explore how different prompt structures, phrasings, and **temperature settings** influence the model's output for each task type. The tutorial also demonstrates best practices for running large models efficiently on free Colab GPUs with 4-bit quantization.

## Conclusion

By the end of this tutorial, you'll have a solid understanding of how to create effective prompts for text summarization, question-answering, code generation, and creative writing tasks using open-source models. You'll be equipped with practical examples and insights that you can apply to your own projects, enhancing your ability to leverage local AI language models for diverse applications. Remember that prompt engineering is both an art and a science — experimentation and iteration are key to finding the most effective prompts for your specific needs.

## Setup

First, let's install dependencies, load the model in 4-bit quantization, and define a reusable `generate()` helper.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## Temperature as Softmax Scaling: The Math

Before diving into task-specific prompts, we need to understand the **single most important generation parameter**: temperature. Temperature controls the *randomness* of token selection, and it does so through a direct mathematical transformation of the model's raw output.

### The Softmax Formula with Temperature

A language model outputs a **logit** (raw score) for every token in its vocabulary (~150,000+ tokens for Qwen3). To convert logits into probabilities, we apply the **temperature-scaled softmax**:

$$p_i = \frac{\exp(z_i \;/\; T)}{\sum_j \exp(z_j \;/\; T)}$$

Where:
- $z_i$ is the logit (raw score) for token $i$
- $T$ is the temperature parameter
- The sum in the denominator runs over **all** tokens in the vocabulary

### What Happens at Different Temperatures?

| Temperature | Effect | Distribution Shape | Use Case |
|---|---|---|---|
| **T → 0** | Dividing by near-zero makes differences enormous | **Spike** — nearly all probability on the highest logit | Deterministic, factual tasks |
| **T = 1.0** | No scaling — model's natural distribution | **Calibrated** — the distribution the model was trained to produce | General-purpose generation |
| **T → ∞** | Dividing by huge number makes all logits ≈ 0 | **Uniform** — every token equally likely | Maximum randomness (rarely useful) |

### Intuition

Think of temperature as a **contrast knob**:
- **Low T** = high contrast: the model's top choice dominates, like turning up the sharpness on a photo
- **High T** = low contrast: differences between choices are washed out, like blurring a photo

The next cell makes this concrete with real numbers from the model.

In [ ]:
import numpy as np

# --- Temperature as Softmax Scaling: Interactive Demonstration ---
# We'll get real logits from the model for a prompt, then show how
# temperature transforms the probability distribution.

demo_prompt = "The capital of France is"
messages = [{"role": "user", "content": demo_prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Get raw logits (no sampling, just the forward pass)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits[0, -1, :]  # logits for the next token

# Move to CPU and convert to numpy for analysis
logits_np = logits.float().cpu().numpy()

# Show the top 10 raw logits so we can see the model's "preferences"
top_indices = np.argsort(logits_np)[::-1][:10]
print("=" * 70)
print("RAW LOGITS — Top 10 tokens the model considers most likely")
print("=" * 70)
for rank, idx in enumerate(top_indices):
    token_str = tokenizer.decode([idx])
    print(f"  #{rank+1:2d}  logit = {logits_np[idx]:8.3f}  token = '{token_str}'")

print("\n" + "=" * 70)
print("TEMPERATURE SCALING — How temperature reshapes the distribution")
print("=" * 70)

temperatures = [0.1, 0.3, 0.7, 1.0, 1.5, 2.0]

for T in temperatures:
    # Apply temperature-scaled softmax: p_i = exp(z_i/T) / sum(exp(z_j/T))
    scaled = logits_np / T
    scaled -= scaled.max()  # numerical stability (subtract max before exp)
    exp_scaled = np.exp(scaled)
    probs = exp_scaled / exp_scaled.sum()

    # Metrics
    top1_prob = probs.max()
    tokens_above_1pct = (probs > 0.01).sum()
    # Entropy: H = -sum(p * log(p)), only for p > 0
    nonzero = probs[probs > 0]
    entropy = -np.sum(nonzero * np.log2(nonzero))

    # Top 5 tokens at this temperature
    top5 = np.argsort(probs)[::-1][:5]
    top5_str = ", ".join(f"'{tokenizer.decode([i])}' ({probs[i]:.1%})" for i in top5)

    print(f"\n  T = {T:.1f}")
    print(f"    Top-1 probability:       {top1_prob:8.4f}  ({top1_prob:.1%})")
    print(f"    Tokens with >1% prob:    {tokens_above_1pct:8d}")
    print(f"    Entropy (bits):          {entropy:8.3f}")
    print(f"    Top 5: {top5_str}")

print("\n" + "=" * 70)
print("KEY TAKEAWAY:")
print("  T=0.1 → almost deterministic (one token dominates)")
print("  T=1.0 → model's natural distribution")
print("  T=2.0 → very spread out (many tokens plausible)")
print("=" * 70)

### What We Just Observed

The demonstration above reveals the core mechanism:

1. **At T=0.1**: The top-1 token captures nearly **100%** of probability. The model is effectively greedy — it will always pick the same token. Entropy is near zero.
2. **At T=1.0**: The model's natural distribution — a handful of tokens share significant probability. This is how the model was trained to behave.
3. **At T=2.0**: Probability is spread across many tokens. Even unlikely tokens get non-trivial probability, leading to surprising (and sometimes incoherent) outputs.

This is the **foundation** for understanding why different tasks need different temperatures — which we explore next.

## Why Different Tasks Need Different Parameters

Now that we understand temperature mathematically, let's build the intuition for **which temperature to use when**.

### The "Answer Space" Framework

Every task has an **answer space** — the set of acceptable responses:

| Task Type | Answer Space | Optimal Temperature | Why? |
|---|---|---|---|
| **Factual Q&A** | Very narrow — usually one correct answer | **0.0 – 0.2** | We want the model's single best guess, with no randomness introducing errors |
| **Classification / Extraction** | Narrow — constrained set of valid outputs | **0.1 – 0.3** | Slight flexibility helps with phrasing, but we need precision |
| **Summarization** | Medium — many valid summaries, but must be faithful | **0.2 – 0.4** | Some variation in wording is fine; hallucination is not |
| **Code Generation** | Medium — many correct implementations exist | **0.2 – 0.5** | Need syntactic correctness, but stylistic variation is acceptable |
| **Translation** | Medium — multiple valid translations possible | **0.3 – 0.5** | Fluency benefits from some variation; accuracy must be preserved |
| **Creative Writing** | Very wide — novelty is the goal | **0.7 – 1.0** | Higher randomness produces more surprising, engaging prose |
| **Brainstorming** | Widest — want diverse, unexpected ideas | **0.8 – 1.2** | Push beyond the model's "safe" defaults to find novel ideas |

### The Rule of Thumb

> **Narrow answer space → Low temperature. Wide answer space → High temperature.**

The next cell demonstrates this empirically: we'll run the same model on tasks with different answer spaces and show what happens when temperature is mismatched.

In [ ]:
# --- Empirical Demonstration: Temperature Mismatch ---
# We'll show what happens when you use the WRONG temperature for a task.

print("=" * 70)
print("EXPERIMENT: Temperature Mismatch Effects")
print("=" * 70)

# --- Factual task at different temperatures ---
factual_messages = [
    {"role": "system", "content": "Answer factual questions in one sentence."},
    {"role": "user", "content": "What is the boiling point of water at sea level in Celsius?"}
]

print("\n--- FACTUAL Q&A: 'What is the boiling point of water?' ---")
for T in [0.1, 0.5, 1.0, 1.5]:
    result = generate(factual_messages, max_new_tokens=80, temperature=T,
                      do_sample=(T > 0.01))
    print(f"  T={T:.1f}: {result.strip()}")

# --- Creative task at different temperatures ---
creative_messages = [
    {"role": "system", "content": "You are a creative writer. Write with imagination and flair."},
    {"role": "user", "content": "Describe a sunset in exactly two sentences."}
]

print("\n--- CREATIVE WRITING: 'Describe a sunset' ---")
for T in [0.1, 0.5, 0.9, 1.3]:
    result = generate(creative_messages, max_new_tokens=120, temperature=T,
                      do_sample=(T > 0.01))
    print(f"  T={T:.1f}: {result.strip()}")

# --- Code task at different temperatures ---
code_messages = [
    {"role": "system", "content": "Write only Python code, no explanation."},
    {"role": "user", "content": "Write a function to check if a string is a palindrome."}
]

print("\n--- CODE GENERATION: 'Palindrome checker' ---")
for T in [0.1, 0.5, 1.0, 1.5]:
    result = generate(code_messages, max_new_tokens=150, temperature=T,
                      do_sample=(T > 0.01))
    # Show just the first 3 lines to keep output compact
    lines = result.strip().split("\n")[:3]
    preview = " | ".join(l.strip() for l in lines)
    print(f"  T={T:.1f}: {preview}")

print("\n" + "=" * 70)
print("OBSERVATIONS:")
print("  • Factual at high T → may introduce errors or hedging language")
print("  • Creative at low T → repetitive, generic, 'safe' phrasing")
print("  • Code at high T → syntax errors, inconsistent naming")
print("  • Each task has an optimal temperature RANGE, not a single value")
print("=" * 70)

### Key Insight: Temperature Controls the Explore/Exploit Tradeoff

This is fundamentally the same tradeoff seen throughout machine learning and decision theory:

- **Low temperature = Exploit**: Use what the model is most confident about. Best when there's a single correct answer.
- **High temperature = Explore**: Sample from a wider range of possibilities. Best when novelty and diversity matter more than precision.

The optimal temperature depends on the **answer space** — narrow answer space → low T, wide answer space → high T. Keep this framework in mind as we work through each task type below.

## Task-Specific Helpers

Different tasks benefit from different generation parameters. Below we define thin wrappers that pair a system prompt with the right temperature and token budget for each task type.

In [ ]:
# Summarization: low temperature, focused output
def summarize(text, max_length=256):
    messages = [
        {"role": "system", "content": "Summarize the text concisely."},
        {"role": "user", "content": text}
    ]
    return generate(messages, max_new_tokens=max_length, temperature=0.3)

# Creative writing: high temperature, diverse output
def creative_write(prompt):
    messages = [
        {"role": "system", "content": "You are a creative fiction writer."},
        {"role": "user", "content": prompt}
    ]
    return generate(messages, max_new_tokens=512, temperature=0.9)

# Code generation: moderate temperature
def generate_code(spec):
    messages = [
        {"role": "system", "content": "You are an expert programmer. Write clean, well-documented code."},
        {"role": "user", "content": spec}
    ]
    return generate(messages, max_new_tokens=1024, temperature=0.2)

# QA: low temperature for factual accuracy
def answer_question(context, question):
    messages = [
        {"role": "system", "content": "Answer the question based only on the provided context."},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
    ]
    return generate(messages, max_new_tokens=256, temperature=0.1)

## Task-Specific Token Budget Analysis

Beyond temperature, the **token budget** (`max_new_tokens`) is the second critical parameter. Setting it too low truncates important content; setting it too high wastes compute and may encourage the model to ramble.

Different tasks naturally produce responses of very different lengths. Let's measure this empirically.

In [ ]:
# --- Token Budget Analysis: How many tokens does each task actually need? ---
# We set a generous max_new_tokens=512 and measure actual usage.

import time

budget_tests = [
    {
        "task": "Summarization",
        "messages": [
            {"role": "system", "content": "Summarize the following text in a concise paragraph."},
            {"role": "user", "content": "Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to natural intelligence displayed by animals including humans. AI research has been defined as the field of study of intelligent agents, which refers to any system that perceives its environment and takes actions that maximize its chance of achieving its goals. The term artificial intelligence had previously been used to describe machines that mimic and display human cognitive skills. AI applications include advanced web search engines, recommendation systems, understanding human speech, self-driving cars, automated decision-making, and competing at the highest level in strategic game systems."}
        ],
        "temp": 0.3
    },
    {
        "task": "Q&A (factual)",
        "messages": [
            {"role": "system", "content": "Answer the question concisely based on the context."},
            {"role": "user", "content": "Context: The Great Wall of China is approximately 21,196 km long and was built over many centuries starting from the 7th century BC.\n\nQuestion: How long is the Great Wall of China?"}
        ],
        "temp": 0.1
    },
    {
        "task": "Creative Writing",
        "messages": [
            {"role": "system", "content": "You are a creative fiction writer."},
            {"role": "user", "content": "Write a short story about a robot discovering music for the first time. About 150 words."}
        ],
        "temp": 0.9
    },
    {
        "task": "Code Generation",
        "messages": [
            {"role": "system", "content": "Write clean, documented Python code."},
            {"role": "user", "content": "Write a function that finds the longest common subsequence of two strings."}
        ],
        "temp": 0.2
    },
    {
        "task": "Data Extraction",
        "messages": [
            {"role": "system", "content": "Extract structured data from the text. Return only the requested fields."},
            {"role": "user", "content": "Extract the person's name, age, and occupation: 'Dr. Sarah Chen, a 42-year-old neurosurgeon at Massachusetts General Hospital, published her groundbreaking research on neural plasticity last Tuesday.'"}
        ],
        "temp": 0.1
    }
]

print("=" * 70)
print("TOKEN BUDGET ANALYSIS — Actual tokens used per task type")
print("=" * 70)
print(f"  {'Task':<22s} {'Tokens Used':>12s} {'Time (s)':>10s}  Preview")
print("-" * 70)

for test in budget_tests:
    t0 = time.time()
    result = generate(test["messages"], max_new_tokens=512, temperature=test["temp"],
                      do_sample=(test["temp"] > 0.01))
    elapsed = time.time() - t0
    token_count = len(tokenizer.encode(result))
    preview = result.strip().replace("\n", " ")[:60]
    print(f"  {test["task"]:<22s} {token_count:>8d}     {elapsed:>7.1f}  {preview}...")

print("-" * 70)
print("\nTYPICAL TOKEN RANGES:")
print("  Summarization:      50 – 150 tokens")
print("  Q&A (factual):      20 – 100 tokens")
print("  Creative Writing:  200 – 500 tokens")
print("  Code Generation:   100 – 300 tokens")
print("  Data Extraction:    30 –  80 tokens")
print("\n→ Match your max_new_tokens to the task to avoid waste or truncation.")

### Token Budget Guidelines

Based on empirical observation:

| Task | Recommended `max_new_tokens` | Reasoning |
|---|---|---|
| Summarization | 128 – 256 | Summaries should be concise; generous budget prevents truncation |
| Factual Q&A | 64 – 128 | Answers are short; tight budget discourages rambling |
| Creative writing | 256 – 512 | Stories need room to develop |
| Code generation | 256 – 1024 | Functions vary widely in length; err on the generous side |
| Data extraction | 64 – 128 | Structured output is compact |

**Pro tip**: Setting `max_new_tokens` too tight can cause the model to produce an abrupt, incomplete response. When in doubt, be slightly generous — you can always post-process to trim.

## 1. Text Summarization Prompts

Let's start with text summarization. We use a low temperature (0.3) so the model stays focused and faithful to the source material.

In [ ]:
# Example text to summarize
long_text = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to natural intelligence displayed by animals including humans.
AI research has been defined as the field of study of intelligent agents, which refers to any system that perceives its environment and takes actions that maximize its chance of achieving its goals.
The term "artificial intelligence" had previously been used to describe machines that mimic and display "human" cognitive skills that are associated with the human mind, such as "learning" and "problem-solving".
This definition has since been rejected by major AI researchers who now describe AI in terms of rationality and acting rationally, which does not limit how intelligence can be articulated.
AI applications include advanced web search engines, recommendation systems, understanding human speech, self-driving cars, automated decision-making and competing at the highest level in strategic game systems.
As machines become increasingly capable, tasks considered to require "intelligence" are often removed from the definition of AI, a phenomenon known as the AI effect.
"""

summary = summarize(long_text)

print("Summary:")
print(summary)

## 2. Question-Answering Prompts

Next, let's tackle question-answering. We use a very low temperature (0.1) to maximise factual accuracy and keep the answer grounded in the provided context.

In [ ]:
# Example context and question
context = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.
It is named after the engineer Gustave Eiffel, whose company designed and built the tower.
Constructed from 1887 to 1889 as the entrance arch to the 1889 World's Fair, it was initially criticized by some of France's leading artists and intellectuals for its design, but it has become a global cultural icon of France and one of the most recognizable structures in the world.
The Eiffel Tower is the most-visited paid monument in the world; 6.91 million people ascended it in 2015.
The tower is 324 metres (1,063 ft) tall, about the same height as an 81-storey building, and the tallest structure in Paris.
"""

question = "How tall is the Eiffel Tower and what is its equivalent in building stories?"

answer = answer_question(context, question)

print("Answer:")
print(answer)

## 3. Code Generation Prompts

Now, let's generate code. We use a low temperature (0.2) so the model produces deterministic, syntactically correct output while still allowing a little variation in style.

In [ ]:
# Example task
task = "Create a Python function that takes a list of numbers and returns the average of the even numbers in the list."

generated_code = generate_code(task)

print("Generated Code:")
print(generated_code)

## 4. Creative Writing Prompts

Finally, let's try creative writing. We use a high temperature (0.9) to encourage imaginative, diverse, and surprising prose.

In [ ]:
# Example inputs
genre = "science fiction"
setting = "a space station orbiting a distant planet"
theme = "the nature of humanity"

prompt = f"Write a short {genre} story set in {setting} that explores the theme of {theme}. The story should be approximately 150 words long."

story = creative_write(prompt)

print("Generated Story:")
print(story)

## 5. Summarization with Compression Ratios

Summarization isn't one-size-fits-all. Depending on your use case, you might want a one-line headline, a paragraph abstract, or a detailed executive summary. Let's see how to control **compression ratio** through prompt design.

In [ ]:
# --- Summarization at Different Compression Ratios ---

source_text = """
The Amazon rainforest, often referred to as the "lungs of the Earth," spans over
5.5 million square kilometers across nine countries in South America. It contains
approximately 390 billion individual trees divided into 16,000 species. The forest
produces about 20% of the world's oxygen and plays a critical role in regulating
global climate patterns. However, deforestation has accelerated dramatically, with
an estimated 17% of the forest lost in the last 50 years. Scientists warn that if
deforestation reaches 20-25%, the forest could reach a tipping point where it begins
to transform into savanna, releasing billions of tons of stored carbon and
dramatically accelerating climate change. Conservation efforts include protected
areas, indigenous land rights, and international agreements like the Amazon Fund.
"""

source_tokens = len(tokenizer.encode(source_text))
print(f"Source text: {source_tokens} tokens\n")

compression_levels = [
    ("One-line headline", "Summarize in exactly one sentence of no more than 15 words.", 40),
    ("Brief abstract", "Write a 2-3 sentence summary capturing the key facts.", 100),
    ("Detailed summary", "Write a comprehensive summary preserving all key statistics and arguments.", 256),
]

for label, instruction, max_tok in compression_levels:
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": source_text}
    ]
    result = generate(messages, max_new_tokens=max_tok, temperature=0.3)
    result_tokens = len(tokenizer.encode(result))
    ratio = source_tokens / max(result_tokens, 1)
    print(f"--- {label} (compression ≈ {ratio:.1f}x, {result_tokens} tokens) ---")
    print(result.strip())
    print()

## 6. Data Extraction: Structured Output from Unstructured Text

Data extraction is one of the most practically valuable LLM tasks: pulling structured, machine-readable data from free-form natural language. The key is specifying the **exact output format** in your prompt.

In [ ]:
# --- Data Extraction: Structured Output from Unstructured Text ---

def extract_data(text, schema_description, output_format="JSON"):
    """Extract structured data from unstructured text."""
    messages = [
        {"role": "system", "content": f"Extract structured data from the text. Return ONLY valid {output_format} with no additional commentary. Schema: {schema_description}"},
        {"role": "user", "content": text}
    ]
    return generate(messages, max_new_tokens=256, temperature=0.1, do_sample=True)

# Example 1: Extract from a news article
article = """
On March 15, 2024, tech giant Nexora Inc. announced the acquisition of CloudMesh
Systems for $4.2 billion. The deal, expected to close by Q3 2024, will bring
CloudMesh's 2,300 employees under Nexora's cloud infrastructure division. CEO
Maria Rodriguez stated that the acquisition will strengthen Nexora's position in
the enterprise cloud market, where CloudMesh holds approximately 8% market share.
"""

schema = "acquirer, target, price, expected_close, employee_count, market_share"
print("--- Extraction from news article ---")
print(extract_data(article, schema))

# Example 2: Extract from a product review
review = """
I bought the XR-500 wireless headphones last week for $79.99 from TechMart.
The battery life is amazing — I got about 35 hours on a single charge. Sound
quality is great for the price, though the bass could be stronger. They're
comfortable for long sessions. I'd give them 4 out of 5 stars. The Bluetooth
5.3 connection is rock solid, never had a dropout.
"""

schema2 = "product_name, price, retailer, battery_life_hours, rating, bluetooth_version, pros, cons"
print("\n--- Extraction from product review ---")
print(extract_data(review, schema2))

## 7. Code Review: Analyzing Code for Bugs and Improvements

LLMs can serve as an always-available code reviewer. The key is structuring the prompt to request **specific categories** of feedback rather than generic comments.

In [ ]:
# --- Code Review: Analyze Code for Bugs and Improvements ---

code_to_review = """
def calculate_average(numbers):
    total = 0
    for i in range(len(numbers)):
        total = total + numbers[i]
    average = total / len(numbers)
    return average

def find_duplicates(lst):
    duplicates = []
    for i in range(len(lst)):
        for j in range(len(lst)):
            if i != j and lst[i] == lst[j]:
                duplicates.append(lst[i])
    return duplicates

def read_config(filename):
    f = open(filename, "r")
    data = f.read()
    config = eval(data)
    return config
"""

review_messages = [
    {"role": "system", "content": """You are an expert code reviewer. Analyze the code and provide feedback in these categories:
1. BUGS: Actual errors that would cause incorrect behavior or crashes
2. SECURITY: Security vulnerabilities
3. PERFORMANCE: Inefficiencies that could be improved
4. STYLE: Pythonic improvements and best practices
Be specific — reference line numbers and provide corrected code snippets."""},
    {"role": "user", "content": f"Review this Python code:\n```python\n{code_to_review}\n```"}
]

# Low temperature for precise, analytical feedback
review = generate(review_messages, max_new_tokens=1024, temperature=0.2)
print("--- Code Review Results ---")
print(review)

## 8. Translation Between Natural Languages

Translation requires a balance between **faithfulness** (preserving meaning) and **fluency** (sounding natural in the target language). A moderate temperature allows natural-sounding phrasing without drifting from the source meaning.

In [ ]:
# --- Translation Between Natural Languages ---

def translate(text, source_lang, target_lang, style="natural"):
    """Translate text between languages with style control."""
    messages = [
        {"role": "system", "content": f"You are a professional translator. Translate from {source_lang} to {target_lang}. Style: {style}. Preserve the original meaning, tone, and any cultural nuances. Return ONLY the translation."},
        {"role": "user", "content": text}
    ]
    return generate(messages, max_new_tokens=512, temperature=0.4)

# Example 1: English to French (formal)
en_text = "The quarterly earnings report shows a 15% increase in revenue, driven primarily by strong performance in the Asian markets."
print("--- English → French (formal) ---")
print(f"Source:  {en_text}")
print(f"French:  {translate(en_text, 'English', 'French', 'formal business')}")

# Example 2: English to Spanish (casual)
casual_text = "Hey, want to grab coffee later? I found this amazing new place downtown that has the best pastries."
print("\n--- English → Spanish (casual) ---")
print(f"Source:   {casual_text}")
print(f"Spanish:  {translate(casual_text, 'English', 'Spanish', 'casual conversational')}")

# Example 3: English to Japanese (polite)
polite_text = "We would like to invite you to our annual conference on artificial intelligence, to be held in Tokyo on September 15th."
print("\n--- English → Japanese (polite) ---")
print(f"Source:    {polite_text}")
print(f"Japanese:  {translate(polite_text, 'English', 'Japanese', 'polite/formal (keigo)')}")

## 9. Technical Documentation Generation

Generating technical documentation requires the model to infer intent, parameters, return values, and edge cases from code — then present them in a clear, structured format. Low temperature ensures consistency and accuracy.

In [ ]:
# --- Technical Documentation Generation ---

code_to_document = """
class RateLimiter:
    def __init__(self, max_requests, window_seconds):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.requests = {}

    def is_allowed(self, client_id):
        import time
        now = time.time()
        if client_id not in self.requests:
            self.requests[client_id] = []
        self.requests[client_id] = [
            t for t in self.requests[client_id]
            if now - t < self.window_seconds
        ]
        if len(self.requests[client_id]) < self.max_requests:
            self.requests[client_id].append(now)
            return True
        return False

    def reset(self, client_id=None):
        if client_id:
            self.requests.pop(client_id, None)
        else:
            self.requests.clear()
"""

doc_messages = [
    {"role": "system", "content": """Generate comprehensive technical documentation for the provided code. Include:
1. A module/class overview
2. Constructor parameters with types and descriptions
3. Method documentation with parameters, return values, and examples
4. Edge cases and important notes
5. A usage example
Use Markdown formatting with proper headings."""},
    {"role": "user", "content": f"Document this Python class:\n```python\n{code_to_document}\n```"}
]

documentation = generate(doc_messages, max_new_tokens=1024, temperature=0.2)
print("--- Generated Technical Documentation ---")
print(documentation)

## Task Difficulty Estimation

Not all prompts are equally hard for the model. A simple factual question and a nuanced ethical dilemma require very different approaches. Can we **estimate task difficulty before generating a full response**?

Two useful heuristics:
1. **Input complexity**: Longer, more complex inputs tend to require more careful generation parameters.
2. **First-token entropy**: If the model is uncertain about even the *first* token of its response, the task is likely ambiguous or difficult — a signal that chain-of-thought prompting or more specific instructions may help.

In [ ]:
# --- Task Difficulty Estimation ---
# Estimate difficulty BEFORE generating a full response.

import numpy as np

def estimate_difficulty(messages):
    """Estimate task difficulty based on input complexity and model confidence."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_tokens = inputs.input_ids.shape[1]

    # Get logits for the first predicted token
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :]

    # Compute probability distribution and entropy
    probs = torch.softmax(logits, dim=-1).float().cpu().numpy()
    nonzero = probs[probs > 0]
    entropy = -np.sum(nonzero * np.log2(nonzero))

    # Top-1 and top-5 probability mass
    sorted_probs = np.sort(probs)[::-1]
    top1_prob = sorted_probs[0]
    top5_prob = sorted_probs[:5].sum()
    tokens_above_1pct = (probs > 0.01).sum()

    return {
        "input_tokens": input_tokens,
        "first_token_entropy": entropy,
        "top1_confidence": top1_prob,
        "top5_confidence": top5_prob,
        "tokens_above_1pct": tokens_above_1pct,
    }

# Test on tasks of varying difficulty
test_tasks = [
    ("Simple factual", [
        {"role": "system", "content": "Answer concisely."},
        {"role": "user", "content": "What is 2 + 2?"}
    ]),
    ("Moderate factual", [
        {"role": "system", "content": "Answer concisely."},
        {"role": "user", "content": "Explain the difference between TCP and UDP protocols."}
    ]),
    ("Complex analysis", [
        {"role": "system", "content": "Provide a thorough analysis."},
        {"role": "user", "content": "Compare and contrast the economic policies of Keynesian and Austrian schools of economics, with specific examples of each being applied in the 21st century."}
    ]),
    ("Ambiguous creative", [
        {"role": "system", "content": "You are a creative writer."},
        {"role": "user", "content": "Write something beautiful."}
    ]),
    ("Constrained creative", [
        {"role": "system", "content": "You are a creative writer."},
        {"role": "user", "content": "Write a haiku about a winter morning."}
    ]),
]

print("=" * 75)
print("TASK DIFFICULTY ESTIMATION — First-token analysis")
print("=" * 75)
print(f"  {'Task':<22s} {'Input Tok':>9s} {'Entropy':>8s} {'Top-1 %':>8s} {'Top-5 %':>8s} {'P>1%':>6s}")
print("-" * 75)

for name, msgs in test_tasks:
    d = estimate_difficulty(msgs)
    print(f"  {name:<22s} {d["input_tokens"]:>9d} {d["first_token_entropy"]:>8.2f} {d["top1_confidence"]:>7.1%} {d["top5_confidence"]:>7.1%} {d["tokens_above_1pct"]:>6d}")

print("-" * 75)
print("\nINTERPRETATION:")
print("  • Low entropy + high top-1 → Model is confident → simple task, low temperature is fine")
print("  • High entropy + low top-1 → Model is uncertain → consider CoT, more specific instructions,")
print("    or break the task into sub-tasks")
print("  • Ambiguous prompts (\"write something beautiful\") have higher entropy than")
print("    constrained ones (\"write a haiku about winter\") — specificity reduces uncertainty")

### Putting It All Together: Adaptive Generation

The difficulty estimation above suggests a practical workflow:

1. **Estimate difficulty** — check first-token entropy before committing to a full generation
2. **Choose parameters** — high confidence → use low temperature and tight token budget; low confidence → consider chain-of-thought, higher temperature, or prompt refinement
3. **Match token budget to task type** — use the empirical ranges from the Token Budget Analysis section
4. **Iterate** — if the output quality is poor, adjust temperature and instructions based on the difficulty signal

This transforms prompt engineering from guesswork into a **data-driven process**: measure the model's confidence, then adapt your strategy accordingly.